# OceanEmbed · STOP 3
Run only after STOP 2 produced a reviewed regional.nc. This notebook trains the provided CNN+TCN and returns an inference bundle to VS Code. No real data or pretrained weights are bundled.

Runtime → Change runtime type → GPU if available. Your checkpoint goes to Drive each completed epoch. Colab resources are variable.


## STOP: upload the master code ZIP
Choose the OceanEmbed code ZIP you downloaded from Codex. Do not upload a dataset ZIP here.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, subprocess, sys
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload exactly one master code ZIP")
root = Path("/content")
with zipfile.ZipFile(archives[0]) as z:
    for info in z.infolist():
        dest = (root / info.filename).resolve()
        if not dest.is_relative_to(root):
            raise ValueError("Unsafe ZIP path")
    z.extractall(root)
project = root / "oceanembed-nio"
assert (project / "pyproject.toml").exists(), "Choose the master code ZIP"
os.chdir(project)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run([sys.executable, "-m", "oceanembed.cli", "doctor"], check=True)


## STOP: mount Drive and upload your prepared cube
Approve Google's own Drive prompt. In Drive's browser upload regional.nc and its regional.sources.json to MyDrive/OceanEmbed/data. Dataset preparation happens before this notebook. Do not substitute raw GLORYS files.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, yaml, json, torch
RUN_NAME = "regional-v1"
drive_root = Path("/content/drive/MyDrive/OceanEmbed")
cube_on_drive = drive_root / "data/regional.nc"
assert cube_on_drive.exists(), "Upload reviewed regional.nc to Drive first"
local_cube = project / "data/processed/regional.nc"
local_cube.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(cube_on_drive, local_cube)
subprocess.run([sys.executable, "-m", "oceanembed.cli", "validate", str(local_cube)], check=True)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CPU runtime: choose a GPU before full regional training if available.")


## Freeze configuration
Check dates against the data reviewed at STOP 2. Use a new RUN_NAME when changing model settings. Reconnecting with the same run preserves its config.


In [ ]:
config = yaml.safe_load((project / "configs/regional.yaml").read_text())
config["cube"] = str(local_cube)
config["output"] = str(drive_root / "runs" / RUN_NAME)
run_dir = Path(config["output"])
run_dir.mkdir(parents=True, exist_ok=True)
frozen = run_dir / "requested_config.yaml"
if frozen.exists():
    saved = yaml.safe_load(frozen.read_text())
    if saved != config:
        raise ValueError("Existing run has different settings. Restore them or use a new RUN_NAME.")
else:
    frozen.write_text(yaml.safe_dump(config))
print(yaml.safe_dump(config))
# Optional: inspect the regional.sources.json you preserved in Drive.
source_manifest = drive_root / "data/regional.sources.json"
if source_manifest.exists():
    shutil.copy2(source_manifest, run_dir / "source_manifest.json")


## Small pilot first
This pilot verifies memory and optimization; it is not a skill benchmark. Skip it on reconnect if it has already passed. If it fails, stop and send the exact error.


In [ ]:
pilot = dict(config)
pilot.update(output=str(drive_root / "pilots" / RUN_NAME), epochs=1, max_batches=3)
pilot_path = project / "configs/colab-pilot.yaml"
pilot_path.write_text(yaml.safe_dump(pilot))
pilot_args = [sys.executable, "-m", "oceanembed.cli", "train", str(pilot_path)]
pilot_resume = Path(pilot["output"]) / "resume.pt"
if pilot_resume.exists():
    pilot_args += ["--resume", str(pilot_resume)]
subprocess.run(pilot_args, check=True)


## Full training or resume
Only continue after the pilot is healthy. Keep the same config when resuming. Load only your own trusted resume checkpoint.


In [ ]:
args = [sys.executable, "-m", "oceanembed.cli", "train", str(frozen)]
resume = run_dir / "resume.pt"
if resume.exists():
    args += ["--resume", str(resume)]
subprocess.run(args, check=True)
history = json.loads((run_dir / "history.json").read_text())
print("Completed epochs:", len(history))
print("Latest record:", history[-1])


## Review and export the selected model
Training loss is not independent Argo validation. Run the STOP 4 commands locally for validation, calibration and final testing. The bundle includes weights, prior/normalization and configuration, not the raw training cube.


In [ ]:
import matplotlib.pyplot as plt
plt.plot([h["train"] for h in history], label="Training crop loss")
plt.plot([h["validation"] for h in history], label="Validation crop loss")
plt.xlabel("Epoch"); plt.ylabel("Depth-balanced MSE"); plt.legend(); plt.show()
bundle = drive_root / ("bundle-" + RUN_NAME)
if not bundle.exists():
    subprocess.run([sys.executable, "-m", "oceanembed.cli", "bundle", str(run_dir), "--out", str(bundle)], check=True)
else:
    raise ValueError("Bundle already exists. Download the existing ZIP or choose a new version after further training.")
archive = shutil.make_archive(str(drive_root / ("bundle-" + RUN_NAME)), "zip", root_dir=drive_root, base_dir=bundle.name)
files.download(archive)


## STOP 4 — return to VS Code
Extract bundle-regional-v1 into your local artifacts folder. Keep regional.nc locally. Follow docs/STOPS.md for inference, Argo matching and API configuration. Do not copy resume.pt into the application or claim the synthetic demo metrics as model accuracy.
